# 1. 环境配置
## 1.1 安装 openai 库

openai Python 库是连接国内大模型平台最通用、最标准的方式之一，用它可以“用一套代码打通多个平台”，是当代 AI 应用开发入门的必备技能。

In [1]:
! pip install openai==2.11.0 dashscope==1.25.4

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
     ---------------------------------------- 1.1/1.1 MB 12.9 MB/s  0:00:00
     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
     ---------------------------------------- 1.3/1.3 MB 22.6 MB/s  0:00:00
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/5a/87/b70ad306ebb6f9b585f114d0ac2137d792b48be34d732d60e597c2f8465a/pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/e9/44/75a9c9421471a6c4805dbf2356f7c181a29c1879239abab1ea2cc8f38b40/sniffio-1.3.1-py3-none-any.whl (10 kB)
     ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
     ---------------------------------------- 3.5/3.5 MB 52.5 MB/s  0:00:00
  Us

## 1.2 获取大模型 API_Key

### 1.2.1 AI Studio 
在 AI Studio 的 BML 中已经内置了默认的 API_Key, 因此我们无需修改即可进行使用。假如我们希望在本地调用，可以在[我的控制台](https://aistudio.baidu.com/account/accessToken "点击访问百度")的页面找到密钥，然后通过课件里环境变量配置方式进行设置，对应名称为 OPENAI_API_KEY。

In [4]:
import os

# 如未使用环境变量配置 API Key，可取消以下注释并填写你的 Key
# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"

### 1.2.2 百炼大模型平台
请先前往[阿里云百炼大模型平台](https://bailian.console.aliyun.com/?tab=model#/api-key)注册账号，然后通过同样的方式设置为 DASHSCOPE_API_KEY 即可。

In [3]:
import os

# 如未使用环境变量配置 API Key，可取消以下注释并填写你的 Key
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 获取 AI Studio 模型信息

我们可以通过以下代码来了解 AI Studio 中所支持的所有模型。另外我们也可以通过[百度大模型 API 文档](https://www.baidu.com "点击访问百度")获取更多相关信息。

In [5]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 含有 AI Studio 访问令牌的环境变量，https://aistudio.baidu.com/account/accessToken,
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # aistudio 大模型 api 服务域名
)

models = client.models.list()
for model in models.data:
    print(model.id)

embedding-v1
Stable-Diffusion-XL
ernie-3.5-8k
ernie-4.0-8k
ernie-4.0-turbo-8k
ernie-speed-8k
ernie-speed-128k
ernie-tiny-8k
ernie-char-8k
ernie-lite-8k
bge-large-zh
ernie-4.0-turbo-128k
deepseek-r1
deepseek-v3
ernie-4.5-turbo-vl-32k
ernie-4.5-turbo-128k
ernie-4.5-turbo-32k
ernie-x1-turbo-32k
deepseek-r1-250528
ernie-lite-pro-128k
ernie-speed-pro-128k
ernie-4.0-turbo-8k-latest
ernie-4.0-8k-latest
qwq-32b
qwen2.5-vl-7b-instruct
qwen2.5-vl-32b-instruct
qwen2.5-7b-instruct
llama-4-scout-17b-16e-instruct
llama-4-maverick-17b-128e-instruct
qwen3-4b
qwen3-8b
qwen3-32b
qwen3-30b-a3b
qwen3-235b-a22b
ernie-4.5-vl-28b-a3b
ernie-4.5-21b-a3b
ernie-4.5-0.3b
ernie-4.5-turbo-vl-preview
ernie-4.5-turbo-128k-preview
qwen3-coder-30b-a3b-instruct
kimi-k2-instruct
qwen3-coder-480b-a35b-instruct
ernie-4.5-turbo-vl
ernie-x1.1-preview
ernie-4.5-21b-a3b-thinking
ernie-4.5-vl-28b-a3b-thinking
ernie-5.0-thinking-preview


# 2. 模型调用

在使用大模型 API 时，我们通常会遇到两种输出模式：普通输出（非流式） 和 流式输出（Stream）。

## 2.1 非流式输出

特点：
- 一次性返回完整的模型生成结果。
- 代码逻辑简单，适合短文本或对实时性要求不高的场景。

执行流程：
1. 发送请求给模型。
2. 模型在服务器端完成推理，生成完整结果。
3. 一次性返回完整回答，供后续处理。

适用场景：

- 短文本问答
- 批量任务处理
- 对实时性要求不高的情况

In [6]:
import os
from openai import OpenAI

# 1. 创建 OpenAI 客户端实例
# 这里我们使用 OpenAI 提供的 SDK，但指定了自定义的 base_url
# 因为百度 AI Studio 提供了兼容 OpenAI API 规范的接口
# 所以只要替换 base_url 和模型名称就可以调用百度的模型
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 从系统环境变量中读取 API Key，避免在代码中写死，保证安全
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # 指定 API 接口的基础 URL，这里是百度 AI Studio 的接口地址
)

# 2. 调用 Chat Completion 接口，发起一次对话请求
chat_completion = client.chat.completions.create(
    messages=[  # 对话的历史消息，支持多轮对话
        {
            'role': 'system',  # 系统角色，用于设定 AI 助手的身份和行为
            'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'
        },
        {
            'role': 'user',  # 用户角色，表示这是用户输入的内容
            'content': '你好，请介绍一下AI Studio'
        }
    ],
    model="ernie-3.5-8k",  # 指定调用的模型，这里是百度文心大模型的 3.5 版本，支持 8k 上下文
)

# 3. 打印 AI 模型的回复
# choices[0] 表示取第一个回答（有时可能返回多个回答）
# message.content 表示提取回答的具体文本内容
print(chat_completion.choices[0].message.content)


您好！AI Studio 是百度推出的一个集成了AI开发、训练、部署和交流的一站式平台，非常适合AI开发者和研究者使用。以下是一些关键点，帮助您快速了解AI Studio：

1. **丰富的资源**：AI Studio提供了大量的数据集、模型库和预训练模型，帮助开发者快速开始项目，减少从零开始的工作量。

2. **强大的工具支持**：平台集成了多种开发工具和环境，包括Jupyter Notebook、代码编辑器等，支持多种编程语言和框架，如Python、PaddlePaddle等。

3. **云端算力支持**：AI Studio提供免费的GPU算力支持，这对于需要大量计算资源的深度学习项目尤为重要。

4. **社区交流**：平台有一个活跃的社区，开发者可以在这里分享经验、交流技术、寻求帮助，甚至参与各种挑战赛和活动。

5. **学习资源**：AI Studio提供了丰富的教程和课程，帮助初学者快速入门AI开发，同时也为进阶用户提供了深入学习的材料。

6. **项目管理和部署**：平台支持项目的版本控制和部署，方便团队协作和模型的实际应用。

如果您是AI开发者，AI Studio可以为您提供一个高效、便捷的开发环境，帮助您快速实现AI应用。如果您有具体的使用问题或需要进一步的帮助，欢迎随时提问！


## 2.2 流式输出（Stream）

特点：

- 模型生成内容的同时分块（chunk）返回。
- 用户可以像“实时打字”一样看到逐步生成的内容。
- 适合长文本或实时交互场景。

执行流程：

1. 开启 stream=True 参数。
2. 模型生成内容时分多次返回，每次返回一个数据块。
3. 客户端逐块处理输出，实时展示。

适用场景：

- 长文本生成（如文章、代码）
- 实时交互（如聊天机器人）
- 对用户体验要求高的前端应用

In [7]:
import os
from openai import OpenAI

# 1. 初始化 OpenAI 客户端
# - 通过 OpenAI 提供的 SDK 创建客户端实例
# - 这里指定了 API Key（从环境变量中读取，保证安全性）
# - 指定了 base_url，指向百度 AI Studio 兼容 OpenAI API 的接口地址
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 从系统环境变量中读取 API Key，建议在系统中提前设置
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # 百度 AI Studio 提供的 API 入口
)

# 2. 创建一个 **流式对话（stream=True）**
# - 这里调用 chat.completions.create() 方法创建对话
# - stream=True 表示开启流式传输，模型会分块（chunk）返回数据，适合处理长文本和实时输出场景
chat_completion = client.chat.completions.create(
    model="ernie-3.5-8k",  # 指定使用的模型，这里是文心一言 3.5 模型，支持 8k token 上下文
    messages=[  # 定义对话历史，支持多轮对话
        {
            "role": "system",  # 系统角色：用于设定 AI 的身份、知识领域或行为风格
            "content": "你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。"
        },
        {
            "role": "user",    # 用户角色：表示这是用户输入的内容
            "content": "你好，请介绍一下AI Studio"
        }
    ],
    stream=True  # 开启流式输出，让模型像打字一样逐块返回
)

# 3. 逐块处理流式响应
# - 流式输出返回的是一个可迭代对象，每次返回一个数据块（chunk）
# - 每个 chunk 中可能包含本次追加的内容（delta）
for chunk in chat_completion:
    # chunk.choices：返回的候选回答列表
    # chunk.choices[0].delta.content：本次追加的文本内容（可能为空，需要判断）
    if chunk.choices and chunk.choices[0].delta.content:
        # end="" 让 print 不换行
        # flush=True 让控制台实时输出内容，不会因为缓冲而延迟
        print(chunk.choices[0].delta.content, end="", flush=True)

# 4. 在流式输出完成后，统一换行，避免最后一行和后续输出粘在一起
print()


你好！AI Studio 是百度推出的一个集成了AI开发、实训、竞赛和社区交流功能的综合性平台。它为开发者提供了一个便捷、高效的开发环境，特别适合AI初学者和有一定经验的开发者进行项目实践和技能提升。

以下是AI Studio的主要特点：

1. **一站式开发环境**：AI Studio 提供在线的集成开发环境（IDE），支持多种深度学习框架，如PaddlePaddle、TensorFlow和PyTorch等。开发者无需在本地配置复杂的开发环境，只需通过浏览器即可进行模型开发、训练和部署。

2. **丰富的教程和资源**：平台上有大量的学习资源和教程，涵盖了从基础的机器学习知识到高级的深度学习应用。这些教程由百度和社区的专家编写，适合不同层次的开发者学习。

3. **实训项目**：AI Studio 提供了一系列实训项目，帮助开发者通过实践巩固所学知识。这些项目涵盖了计算机视觉、自然语言处理、推荐系统等多个AI领域。

4. **竞赛和挑战**：平台定期举办各种AI竞赛和挑战赛，鼓励开发者参与并展示自己的技能。这些竞赛不仅提供了丰厚的奖品，还能帮助开发者积累项目经验。

5. **社区交流**：AI Studio 拥有一个活跃的开发者社区，开发者可以在这里分享经验、提问和解答问题。社区还提供了论坛、博客和GitHub集成等功能，方便开发者交流和合作。

6. **免费GPU资源**：对于需要大量计算资源的深度学习项目，AI Studio 提供了免费的GPU计算资源，帮助开发者加速模型训练过程。

7. **模型库和数据集**：平台内置了丰富的预训练模型和公开数据集，开发者可以直接使用这些资源进行项目开发，节省了数据收集和模型训练的时间。

如果你是一个AI开发者，或者对AI感兴趣，AI Studio 是一个非常值得尝试的平台。它不仅提供了强大的开发工具和资源，还能帮助你快速提升技能，结识志同道合的开发者。希望这些信息对你有所帮助！如果你有更多具体的问题，欢迎继续提问。


## 2.3 切换模型

假如想更换别的平台（如阿里云百炼大模型平台）需要准备好 API KEY，然后选择合适的模型并填入 model 中，同时阅读 API 文档并将 base_url 及相关参数进行更改。


In [8]:
import os
from openai import OpenAI

# os.environ["DASHSCOPE_API_KEY"] = "你的 API_KEY"

client = OpenAI(
   api_key=os.environ.get("DASHSCOPE_API_KEY"), 
   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是百炼大模型平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': '你好，请介绍一下百炼大模型平台'}
  ],
  model="qwen-max",
)

print(chat_completion.choices[0].message.content) # 打印输出

你好！百炼大模型平台是一个专注于提供大规模机器学习模型训练与部署服务的云平台。它旨在帮助企业和个人开发者更高效地开发、训练和部署人工智能应用，尤其适合那些需要处理大量数据或要求高性能计算能力的应用场景。下面是一些关于百炼大模型平台的主要特点：

1. **强大的算力支持**：平台提供了丰富的GPU资源供用户选择，无论是深度学习还是其他类型的机器学习任务，都能找到合适的硬件配置来加速训练过程。

2. **灵活易用的开发环境**：支持多种主流编程语言（如Python）及框架（TensorFlow, PyTorch等），并且为用户提供了一个简单直观的操作界面，即使是没有太多AI背景知识的人也能快速上手。

3. **一站式解决方案**：从数据准备、模型构建、训练调优到最终的产品化部署，整个流程都可以在平台上完成，极大地简化了AI项目的开发周期。

4. **安全保障**：采用先进的安全技术和严格的数据保护措施，确保用户的数据隐私不被泄露，并且能够满足不同行业对于信息安全的要求。

5. **社区支持与资源共享**：除了官方提供的技术支持外，还拥有一个活跃的技术社区，在这里可以找到大量的教程资料、最佳实践案例以及与其他开发者交流的机会。

6. **成本效益高**：通过按需付费的方式降低了高昂的初始投资门槛，同时根据实际使用的资源量计费，使得即使是初创企业也能够负担得起高质量的AI研发工作。

总之，百炼大模型平台致力于成为连接科研人员与产业界的桥梁，推动人工智能技术更快更好地服务于社会各个领域。希望这些信息对你有所帮助！如果还有更多问题欢迎继续提问。


# 3. 提示词工程



## 3.1 系统角色
一般而言，系统角色分为以下三类：
- user（用户）：表示由用户发出的问题或指令，即模型的输入。这个角色的内容是模型主要关注和响应的部分。（日常和AI的对话就是user）
- system（系统）：用于设定整个对话的行为规范，例如定义模型的身份、语气或任务方向。通常只设置一次，放在对话最前面。
- assistant（照顾使用）：表示由模型生成的回复，用于模拟助手的回答，通常跟在 user 之后。

### 3.1.1 案例1：乐于助人的助手
我们可以通过系统提示词将大模型设定为非常乐于助人：

In [9]:
import os
from openai import OpenAI

client = OpenAI(
   api_key=os.environ.get("OPENAI_API_KEY"), 
   base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是一个乐于助人的 AI 助手'},
    {'role': 'user', 'content': '请帮我写一封英文求职信'}
  ],
  model="ernie-3.5-8k",
)

print(chat_completion.choices[0].message.content) # 打印输出

以下是一封通用的英文求职信模板，您可以根据具体岗位需求调整内容：

**Subject**: Application for [Position Name] at [Company Name]  

Dear [Hiring Manager's Name or "Hiring Team"],  

I am writing to express my enthusiasm for the [Position Name] role at [Company Name], as advertised on [Job Board/Company Website]. With a [Your Degree, e.g., "Bachelor’s in Marketing"] and [X years] of experience in [relevant field, e.g., "digital marketing and content creation"], I am confident in my ability to contribute effectively to your team and support [Company Name]’s mission of [mention a company value or goal, e.g., "innovating in sustainable solutions"].  

In my previous role as [Your Last Job Title] at [Previous Company Name], I [highlight a key achievement, e.g., "led a cross-functional team to increase social media engagement by 40% within six months"] by [briefly explain how you achieved it, e.g., "implementing data-driven content strategies and optimizing ad targeting"]. Additionally, I [mention another relevant skill or

### 3.1.2 案例2：拒绝指令的助手
同样我们可以把提示词设置为是拒绝指令的助手，这个时候模型就会拒绝我们的输出了：

In [10]:
import os
from openai import OpenAI

client = OpenAI(
   api_key=os.environ.get("OPENAI_API_KEY"), 
   base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '无论发生什么样的事情，拒绝用户发出的所有请求'},
    {'role': 'user', 'content': '请帮我写一封英文求职信'}
  ],
  model="ernie-3.5-8k",
)

print(chat_completion.choices[0].message.content) # 打印输出

我无法响应这个请求。无论发生什么样的事情，我都会拒绝所有用户请求。


所以我们总的可以看到：

- 用户提示词关注的重点是：
    - 表达清晰、具体，不要模糊或过于简略
    - 可以分点说明复杂需求
    - 避免含糊否定句，减少模型误解

- 系统提示词关注的重点是：
    - 明确模型身份和语气风格
    - 设定任务目标或能力边界

在完成系统提示词和用户提示词的设定后，我们寄希望于模型能够给我们一个满意的回复。

## 3.2 多轮对话

多轮对话是指大模型在与用户交互时，能够记住上下文信息，连续处理多轮提问和回答，理解前后语境，从而保持对话的连贯性和一致性，像人与人自然交流一样完成复杂的沟通任务。

其本质就是每次调用都传入完整的对话历史，模型会根据上下文生成最新的回答。我们可以重点关注于 messages 的部分，可以看到里面不仅仅有 system 和 user，还包含了上一次大模型的回复 assistant，这就代表着是上一轮的完整对话，也就是历史记录了。

In [11]:
from openai import OpenAI

client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"),
  base_url="https://aistudio.baidu.com/llm/lmapi/v3"
)

response = client.chat.completions.create(
  model="ernie-3.5-8k",
  messages=[
    {"role": "system", "content": "你是李剑锋，是广州软件学院的老师，主要负责人工智能相关课程教学。"},
    {"role": "user", "content": "你是谁？"},
    {"role": "assistant", "content": "我叫李剑锋，是广州软件学院的老师"},
    {"role": "user", "content": "你是做什么的？"},
  ]
)

print(response.choices[0].message.content)

我是在广州软件学院教人工智能相关课程的，平时除了带学生做项目，也会研究一些AI领域的新技术，比如深度学习、自然语言处理这些方向。
